# Required Task 1 (5)


### Instructions

Load the file financial_news.csv.

Last part of the sentence in each row of text contains an url. Remove this from text and create new column called URL and add the url.
Create sentence embeddings for the modified column text. Using Gradio, build a semantic search tool where the user enters some text (such as (“earnings surprise”, “regulatory fine”), and the top 5 closest records (based on cosine similarity) are output to the user.

Source: L3_Embeddings_Words_To_Sentences.ipynb

In [1]:
# imports
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

In [2]:
# upload financial_news.csv
df = pd.read_csv('financial_news.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (16990, 2)
Columns: ['text', 'label']


,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


In [3]:
# Extract URLs from text
# Regex that matches a URL anywhere at the END of the string (after optional whitespace)
URL_PATTERN = re.compile(r'\s*(https?://\S+)\s*$')

def split_text_and_url(raw_text):
    """Return (clean_text, url) by removing the trailing URL from raw_text."""
    match = URL_PATTERN.search(str(raw_text))
    if match:
        url = match.group(1)
        clean = raw_text[:match.start()].strip()
        return clean, url
    return str(raw_text).strip(), None

# Apply to the 'text' column
df[['text', 'URL']] = df['text'].apply(
    lambda t: pd.Series(split_text_and_url(t))
)

print("Sample cleaned text:\n", df['text'].iloc[0])
print("\nSample URL:\n", df['URL'].iloc[0])
df[['text', 'URL']].head()

Sample cleaned text:
 Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more

Sample URL:
 https://t.co/QPN8Gwl7Uh


,text,URL
0,Here are Thursday's biggest analyst calls: App...,https://t.co/QPN8Gwl7Uh
1,Buy Las Vegas Sands as travel to Singapore bui...,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, cit...",https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, bre...",https://t.co/kwhoE6W06u
4,Netflix and its peers are set for a ‘return to...,https://t.co/jPpdl0D9s4


In [5]:
# Generate sentence embeddings

# Load the same pre-trained sentence-transformer model used in the notebook
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all cleaned headlines — this produces a (N, 384) matrix
print("Generating sentence embeddings — this may take a moment...")
corpus_embeddings = sentence_model.encode(
    df['text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbedding matrix shape: {corpus_embeddings.shape}")
print(f"Each sentence is represented as a {corpus_embeddings.shape[1]}-dimensional vector")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating sentence embeddings — this may take a moment...


Batches:   0%|          | 0/531 [00:00<?, ?it/s]


Embedding matrix shape: (16990, 384)
Each sentence is represented as a 384-dimensional vector


Define semantic search function

Given a user query, we:
1. Embed the query with the same model
2. Compute cosine similarity against all corpus embeddings
3. Return the top 5 matches

In [6]:
def semantic_search(query: str) -> pd.DataFrame:
    """
    Encode the query, compute cosine similarity with all corpus embeddings,
    and return the top 5 most similar records as a DataFrame.
    """
    if not query.strip():
        return pd.DataFrame({'Message': ['Please enter a search query.']})

    # Step 1: Embed the query (same model, same vector space)
    query_embedding = sentence_model.encode([query], convert_to_numpy=True)

    # Step 2: Cosine similarity — shape (1, N)
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Step 3: Get indices of top 5 results (descending order)
    top5_indices = np.argsort(similarities)[::-1][:5]

    # Step 4: Build a results DataFrame
    results = df.iloc[top5_indices][['text', 'URL']].copy()
    results.insert(0, 'Similarity', similarities[top5_indices].round(4))
    results.reset_index(drop=True, inplace=True)
    results.index += 1  # Start ranking from 1
    results.index.name = 'Rank'

    return results

# Quick sanity check
test_results = semantic_search("earnings surprise")
print("Test query: 'earnings surprise'")
test_results

Test query: 'earnings surprise'


,Similarity,text,URL
Rank,,,
1,0.6026,SaaS Companies Earnings Are Coming: What To Ex...,None
2,0.5983,Earnings season has begun! table from @eWhispers,https://t.co/UpQJdKM22x
3,0.5750,Wall Street Breakfast: Earnings Evaluation. h...,None
4,0.5731,@equitydd I expect a lot of volatility and wid...,None
5,0.5658,Finding the next surprising earnings winner li...,https://t.co/n1r0i7M0Fo


Building the Gradio Semantic Search Tool

We create an interactive interface where the user types a query (e.g. `"earnings surprise"` or `"regulatory fine"`) and receives the top 5 matching financial news headlines with their similarity scores and source URLs.

In [7]:
# --- Gradio Interface ---

with gr.Blocks(title="Financial News Semantic Search") as demo:

    gr.Markdown("""
    # 📰 Financial News Semantic Search
    Enter a topic or phrase and retrieve the **top 5 most semantically similar**
    financial news headlines, ranked by cosine similarity.

    *Powered by `all-MiniLM-L6-v2` sentence embeddings*
    """)

    with gr.Row():
        query_box = gr.Textbox(
            label="Search Query",
            placeholder='e.g. "earnings surprise" or "regulatory fine"',
            lines=1,
            scale=4
        )
        search_btn = gr.Button("🔍 Search", variant="primary", scale=1)

    gr.Examples(
        examples=[
            ["earnings surprise"],
            ["regulatory fine"],
            ["merger acquisition deal"],
            ["stock market crash"],
            ["interest rate hike"],
            ["CEO resignation"],
        ],
        inputs=query_box,
        label="Example Queries"
    )

    results_table = gr.Dataframe(
        label="Top 5 Matching Headlines",
        headers=["Similarity", "Headline (text)", "URL"],
        wrap=True
    )

    # Wire up the search button and Enter key
    search_btn.click(fn=semantic_search, inputs=query_box, outputs=results_table)
    query_box.submit(fn=semantic_search, inputs=query_box, outputs=results_table)

    gr.Markdown("""
    ---
    **Similarity score** ranges from 0 (unrelated) to 1 (identical meaning).
    Scores above 0.4 typically indicate strong semantic relevance.
    """)

demo.launch(share=True)  # share=True generates a public link in Colab

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://239f37f58045f19e8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
